# Régler le taux, choisir l’arrêt, limiter un gradient

Trois interventions distinctes sur une boucle déjà comprise.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#scheduler-et-arret-anticipe).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


La validation cesse de s’améliorer : faut-il réduire le taux ou arrêter ? Une autre difficulté, un gradient trop grand, appelle encore une opération différente. Ces interventions ne sont pas interchangeables.

Le scheduler StepLR choisi ici multiplie le taux par 0,5 toutes les deux époques. Il ne regarde pas la perte de validation. Son appel vient après les mises à jour et prépare le taux suivant.

Pour isoler l’arrêt anticipé, imposons des validations fictives : 0,90 ; 0,60 ; 0,62 ; 0,65 ; 0,70. Elles ne sont pas mesurées sur le petit poids du programme. En projet, elles seraient remplacées par les pertes des exemples réservés.

La patience vaut deux époques sans amélioration stricte. Nous nous arrêtons à l’époque 4, mais le meilleur état reste celui de l’époque 2. detach() coupe le suivi du graphe ; clone() copie les valeurs. Leur combinaison en garde un instantané ; conserver seulement une référence au poids courant ne le figerait pas.

| Époque | Taux utilisé | Validation fictive | Attente |
| --- | --- | --- | --- |
| 1 | 0,10 | 0,90 | 0 |
| 2 | 0,10 | 0,60 | 0 |
| 3 | 0,05 | 0,62 | 1 |
| 4 | 0,05 | 0,65 | 2 : arrêt |

L’état retenu est le minimum de validation. Le taux change selon les époques, même si la validation se dégrade.

## Démonstration 1 · Suivre les décisions puis revenir au meilleur état

Le calcul sur le poids est réel ; la suite de validations est volontairement imposée.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch


### Calculer l’état suivant

La patience sera de deux époques. La première validation finie améliore l’infini initial.

In [4]:
poids = torch.tensor(1., requires_grad=True)
optimiseur = torch.optim.SGD([poids], lr=0.1)
scheduler = torch.optim.lr_scheduler.StepLR(optimiseur, step_size=2, gamma=0.5)
validations_fictives = [0.9, 0.6, 0.62, 0.65, 0.7]


In [5]:
observer('poids', poids)
observer('scheduler', scheduler)
observer('validations_fictives', validations_fictives)

poids forme () type torch.float32 requires_grad True
  valeurs : 1.0
validations_fictives = [0.9, 0.6, 0.62, 0.65, 0.7]


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [6]:
meilleure = float("inf")
attente = 0
meilleur_poids = poids.detach().clone()
historique = []


In [7]:
observer('meilleure', meilleure)
observer('attente', attente)
observer('meilleur_poids', meilleur_poids)
observer('historique', historique)

meilleure = inf
attente = 0
meilleur_poids forme () type torch.float32 requires_grad False
  valeurs : 1.0
historique = []


### Répéter la même étape sur les données

Le taux est lu avant step pour décrire celui réellement utilisé.

Une amélioration remet l’attente à zéro et copie le poids courant.

Le calendrier modifie le taux ; la patience arrête la boucle.

In [8]:
for epoque, validation in enumerate(validations_fictives, start=1):
    optimiseur.zero_grad()
    perte = (poids - 2.) ** 2
    perte.backward()
    taux = optimiseur.param_groups[0]["lr"]
    optimiseur.step()
    if validation < meilleure:
        meilleure = validation
        meilleur_poids = poids.detach().clone()
        attente = 0
    else:
        attente += 1
    historique.append((epoque, taux, validation, attente))
    scheduler.step()
    if attente >= 2:
        break


### Appliquer le contexte explicite

La copie de la meilleure époque remplace la valeur finale.

In [9]:
with torch.no_grad():
    poids.copy_(meilleur_poids)


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [10]:
print('epoque, taux, validation, attente :', historique)
print('poids restaure :', round(poids.item(), 2))

epoque, taux, validation, attente : [(1, 0.1, 0.9, 0), (2, 0.1, 0.6, 0), (3, 0.05, 0.62, 1), (4, 0.05, 0.65, 2)]
poids restaure : 1.36


In [11]:
assert bool(len(historique) == 4 and historique[-1][0] == 4 and attente == 2), 'scheduler-et-arret-anticipe/main: len(historique) == 4 and historique[-1][0] == 4 and attente == 2'
assert bool([taux for _, taux, _, _ in historique] == [0.1, 0.1, 0.05, 0.05]), 'scheduler-et-arret-anticipe/main: [taux for _, taux, _, _ in historique] == [0.1, 0.1, 0.05, 0.05]'
assert bool(meilleure == 0.6 and abs(poids.item() - 1.36) < 1e-6), 'scheduler-et-arret-anticipe/main: meilleure == 0.6 and abs(poids.item() - 1.36) < 1e-6'
print("Contrôles numériques de la démonstration réussis.")

Contrôles numériques de la démonstration réussis.


Réduire le taux, arrêter et restaurer le meilleur état répondent à trois décisions explicites.

### Si le problème est la taille du gradient

Le clipping agit sur les gradients, pas sur la valeur de la perte. Fabriquons un gradient de 10 avec L=10×poids : cette fonction n’a pas de minimum fini ; elle sert uniquement à isoler l’opération.

clip_grad_norm_ modifie les gradients en place et retourne leur norme avant limitation. Avec un scalaire, max_norm=1 limite sa valeur absolue à environ 1 ; le pas conduit alors à environ 0,9. Sur plusieurs paramètres, la limite porte sur leur norme globale, pas séparément sur chaque case.

## Démonstration 2 · Mesurer le gradient réellement utilisé

Limiter un gradient déjà calculé, sans accumulation ni précision mixte.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [12]:
import torch


### Calculer l’état suivant

La perte affine est choisie pour produire une dérivée constante.

In [13]:
poids = torch.tensor(1., requires_grad=True)
optimiseur = torch.optim.SGD([poids], lr=0.1)
perte = 10 * poids


In [14]:
observer('poids', poids)
observer('perte', perte)

poids forme () type torch.float32 requires_grad True
  valeurs : 1.0
perte forme () type torch.float32 requires_grad True
  valeurs : 10.0


### Calculer les gradients

Le gradient existe maintenant dans poids.grad.

In [15]:
perte.backward()


In [16]:
observer_entrainement({'poids': poids, 'optimiseur': optimiseur, 'perte': perte}, montrer_poids=False)

poids.grad forme () type torch.float32 requires_grad False
  valeurs : 10.0


### Calculer l’état suivant

La fonction retourne la norme ancienne tout en modifiant le gradient en place.

In [17]:
norme_avant = torch.nn.utils.clip_grad_norm_([poids], max_norm=1.)
gradient_utilise = poids.grad.item()


In [18]:
observer('norme_avant', norme_avant)
observer('gradient_utilise', gradient_utilise)

norme_avant forme () type torch.float32 requires_grad False
  valeurs : 10.0
gradient_utilise = 0.9999999403953552


### Appliquer une mise à jour

SGD soustrait environ 0,1 au poids.

In [19]:
optimiseur.step()


In [20]:
observer_entrainement({'poids': poids, 'optimiseur': optimiseur, 'perte': perte, 'norme_avant': norme_avant, 'gradient_utilise': gradient_utilise}, montrer_poids=True)

poids forme () type torch.float32 requires_grad True
  valeurs : 0.8999999761581421
poids.grad forme () type torch.float32 requires_grad False
  valeurs : 0.9999999403953552


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [21]:
print('norme avant :', norme_avant.item())
print('gradient utilise :', round(gradient_utilise, 6))
print('poids apres :', round(poids.item(), 6))

norme avant : 10.0
gradient utilise : 1.0
poids apres : 0.9


In [22]:
assert bool(norme_avant.item() == 10), 'scheduler-et-arret-anticipe/clipping: norme_avant.item() == 10'
assert bool(abs(gradient_utilise - 1) < 1e-6), 'scheduler-et-arret-anticipe/clipping: abs(gradient_utilise - 1) < 1e-6'
assert bool(abs(poids.item() - 0.9) < 1e-6), 'scheduler-et-arret-anticipe/clipping: abs(poids.item() - 0.9) < 1e-6'
print("Contrôles numériques de la démonstration réussis.")

Contrôles numériques de la démonstration réussis.


Placer le clipping avant backward laisserait passer le gradient calculé ensuite.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

StepLR multiplie le taux par 0,5 toutes les deux époques. Quel taux est prêt après les deux premières mises à jour, depuis 0,1 ?

In [23]:
exo_p = torch.nn.Parameter(torch.tensor(1.))
exo_opt = torch.optim.SGD([exo_p],lr=.1)
exo_scheduler = torch.optim.lr_scheduler.StepLR(exo_opt,step_size=2,gamma=.5)
for _ in range(2):
    exo_opt.zero_grad()
    (exo_p**2).backward()
    exo_opt.step()
    exo_scheduler.step()
exo_taux = ...  # Remplacer les points par votre réponse.
if exo_taux is Ellipsis or exo_taux is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_taux)-exo_opt.param_groups[0]["lr"]) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

Le scheduler est appelé après les mises à jour ; le nouveau taux servira à la troisième époque.

</details>

### Exercice 2

Un gradient scalaire vaut 10. Limitez sa norme à 1 avant le pas SGD de taux 0,1. Complétez avec l’appel de clipping.

In [24]:
exo_p=torch.tensor(1.,requires_grad=True)
exo_opt=torch.optim.SGD([exo_p],lr=.1)
(10*exo_p).backward()
exo_norme_avant = ...  # Remplacer les points par votre réponse.
if exo_norme_avant is Ellipsis or exo_norme_avant is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_norme_avant.item() == 10
    assert abs(exo_p.grad.item()-1) < 1e-6
    exo_opt.step()
    assert abs(exo_p.item()-.9) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Le clipping modifie les gradients déjà calculés et renvoie leur norme précédente. Il vient avant step.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_p = torch.nn.Parameter(torch.tensor(1.))
exo_opt = torch.optim.SGD([exo_p],lr=.1)
exo_scheduler = torch.optim.lr_scheduler.StepLR(exo_opt,step_size=2,gamma=.5)
for _ in range(2):
    exo_opt.zero_grad()
    (exo_p**2).backward()
    exo_opt.step()
    exo_scheduler.step()
exo_taux = 0.05
if exo_taux is Ellipsis or exo_taux is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_taux)-exo_opt.param_groups[0]["lr"]) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_p=torch.tensor(1.,requires_grad=True)
exo_opt=torch.optim.SGD([exo_p],lr=.1)
(10*exo_p).backward()
exo_norme_avant = torch.nn.utils.clip_grad_norm_([exo_p],max_norm=1.)
if exo_norme_avant is Ellipsis or exo_norme_avant is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_norme_avant.item() == 10
    assert abs(exo_p.grad.item()-1) < 1e-6
    exo_opt.step()
    assert abs(exo_p.item()-.9) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [27]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()